In [1]:
import os
PROJECT_PATH = "/mnt/c/Users/Yeray/Desktop/MUIA/Computer Vision/COMPUTERVISIONPROYECT"
XVIEW_RECOGNITION_PATH = os.path.join(PROJECT_PATH, "data/xview_detection")
DATASET_NAME = 'stratified'

LOG_NAME = '_'.join([DATASET_NAME,'log.json'])
TRAIN_NAME = '_'.join([DATASET_NAME,'train.tfrecord'])
VALIDATION_NAME = '_'.join([DATASET_NAME,'validation.tfrecord'])

In [2]:
import tensorflow as tf
import numpy as np
import os
import json
import uuid
from tqdm import tqdm
from sklearn.model_selection import train_test_split

JSON_FILE = os.path.join(XVIEW_RECOGNITION_PATH, 'xview_det_train.json')

# Mapping categories to IDs
categories = {
    0: 'Small car', 1: 'Bus', 2: 'Truck', 3: 'Building'
}
id_to_name = {v: k for k, v in categories.items()}

2025-12-14 10:26:30.316427: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
id_to_name

{'Small car': 0, 'Bus': 1, 'Truck': 2, 'Building': 3}

In [4]:
# -----------------------------
# Generic classes
# -----------------------------
class GenericObject:
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category = -1

class GenericImage:
    def __init__(self, filename, width, height):
        self.filename = filename
        self.width = width
        self.height = height
        self.objects = []

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [5]:
with open(JSON_FILE) as f:
    json_data = json.load(f)

    print(json_data.keys())
    print(f"Num images: {len(json_data['images'])}")
    print(f"Num annotations: {len(json_data['annotations'])}")
    print(f"Num categories: {len(json_data['categories'])}")
    print(f"json_data['images']['0']: {json_data['images']['0']}")
    print(f"json_data['annotations']['0']: {json_data['annotations']['0']}")
    print(f"json_data['categories']['0']: {json_data['categories']['0']}")

dict_keys(['info', 'images', 'annotations', 'categories'])
Num images: 7606
Num annotations: 481112
Num categories: 4
json_data['images']['0']: {'image_id': '2511_66439541-a371-4b68-93c8-8c62da2cd64e.tif', 'filename': 'xview_train/2511_66439541-a371-4b68-93c8-8c62da2cd64e.tif', 'num_objects': 9, 'width': 640, 'height': 640}
json_data['annotations']['0']: {'image_id': '2511_66439541-a371-4b68-93c8-8c62da2cd64e.tif', 'category_id': 'Building', 'bbox': [562, 489, 663, 517]}
json_data['categories']['0']: {'id': 18, 'name': 'Small car', 'supercategory': 'Passenger vehicle'}


In [5]:
TILE_SIZE = 512        # 512×512 tiles
STRIDE = 256           # overlap
MIN_BOX_SIZE = 4       # drop boxes smaller than 4 pixels

In [7]:
import json
import os
import random
import shutil
from PIL import Image

JSON_PATH = "data/xview_detection/xview_det_train.json"
IMAGE_ROOT = "data/xview_detection"
OUT_ROOT = "data/dataset_yolo_tiled"
TRAIN_RATIO = 0.8

# -------------------------
# Load JSON
# -------------------------
with open(JSON_PATH) as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]
categories = data["categories"]

print("Images:", len(images))
print("Annotations:", len(annotations))
print("Categories:", len(categories))

# -------------------------
# Class mapping
# -------------------------
class_names = sorted({c["name"] for c in categories.values()})
class_to_id = {name: i for i, name in enumerate(class_names)}
print("Class map:", class_to_id)

# -------------------------
# Image metadata
# -------------------------
image_info = {
    img["image_id"]: img
    for img in images.values()
}

# -------------------------
# Group annotations
# -------------------------
by_image = {}

for ann in annotations.values():
    img_id = ann["image_id"]
    cls = class_to_id[ann["category_id"]]
    xmin, ymin, xmax, ymax = ann["bbox"]

    by_image.setdefault(img_id, []).append(
        (cls, xmin, ymin, xmax, ymax)
    )

print("Images with annotations:", len(by_image))

# -------------------------
# Train / val split
# -------------------------
image_ids = list(by_image.keys())
random.shuffle(image_ids)

split = int(len(image_ids) * TRAIN_RATIO)
train_ids = image_ids[:split]
val_ids = image_ids[split:]

# -------------------------
# Output dirs
# -------------------------
for s in ["train", "val"]:
    os.makedirs(f"{OUT_ROOT}/images/{s}", exist_ok=True)
    os.makedirs(f"{OUT_ROOT}/labels/{s}", exist_ok=True)

# -------------------------
# Write YOLO files
# -------------------------
def write_split(ids, split):
    for img_id in ids:
        info = image_info[img_id]
        src_img = os.path.join(IMAGE_ROOT, info["filename"])

        if not os.path.exists(src_img):
            continue

        img = Image.open(src_img)
        W, H = img.size

        tile_id = 0

        for ty in range(0, H, STRIDE):
            for tx in range(0, W, STRIDE):
                if tx + TILE_SIZE > W or ty + TILE_SIZE > H:
                    continue

                tile = img.crop((tx, ty, tx + TILE_SIZE, ty + TILE_SIZE))

                tile_labels = []

                for cls, xmin, ymin, xmax, ymax in by_image[img_id]:
                    # Intersection with tile
                    ixmin = max(xmin, tx)
                    iymin = max(ymin, ty)
                    ixmax = min(xmax, tx + TILE_SIZE)
                    iymax = min(ymax, ty + TILE_SIZE)

                    if ixmax <= ixmin or iymax <= iymin:
                        continue

                    bw = ixmax - ixmin
                    bh = iymax - iymin

                    if bw < MIN_BOX_SIZE or bh < MIN_BOX_SIZE:
                        continue

                    # Convert to tile-relative YOLO format
                    cx = ((ixmin + ixmax) / 2 - tx) / TILE_SIZE
                    cy = ((iymin + iymax) / 2 - ty) / TILE_SIZE
                    bw /= TILE_SIZE
                    bh /= TILE_SIZE

                    tile_labels.append(
                        f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
                    )

                # Skip empty tiles (optional, but recommended)
                if len(tile_labels) == 0:
                    continue

                tile_name = img_id.replace(
                    ".tif", f"_tile_{tile_id}.jpg"
                )

                tile_path = f"{OUT_ROOT}/images/{split}/{tile_name}"
                label_path = f"{OUT_ROOT}/labels/{split}/{tile_name.replace('.jpg','.txt')}"

                tile.save(tile_path, "JPEG", quality=95)

                with open(label_path, "w") as f:
                    f.write("\n".join(tile_labels))

                tile_id += 1

# -------------------------
# Run
# -------------------------
write_split(train_ids, "train")
write_split(val_ids, "val")

print("✅ Conversion finished")



Images: 7606
Annotations: 481112
Categories: 4
Class map: {'Building': 0, 'Bus': 1, 'Small car': 2, 'Truck': 3}
Images with annotations: 7606
✅ Conversion finished


In [ ]:
pip install ultralytics

In [6]:
!yolo detect train model=yolov8s.pt data=xview_tiled.yaml imgsz=512 batch=16 epochs=100

Ultralytics 8.3.237 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=xview_tiled.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

In [7]:
import os
from ultralytics import YOLO

# Paths
TEST_DIR = "data/xview_detection/xview_test"
MODEL_PATH = "runs/detect/train2/weights/best.pt"
OUTPUT_JSON = "experiment_results/Detection/3_YOLO_tiled_overlap/prediction.json"

# Load trained YOLOv8 model
model = YOLO(MODEL_PATH)

# Categories (must match the training class names)
categories = {
    0: "Building",
    1: "Bus",
    2: "Small car",
    3: "Truck"
}

# Prepare JSON structure
predictions_data = {"images": {}, "annotations": {}, "categories": categories}
imgs_idx, annos_idx = 0, 0

# Iterate over test images
for img_file in os.listdir(TEST_DIR):
    if not img_file.lower().endswith((".jpg", ".png", ".tif")):
        continue

    img_path = os.path.join(TEST_DIR, img_file)
    
    # Run prediction
    results = model.predict(img_path, verbose=False)
    
    # YOLOv8 returns a list of results (one per image), we have only 1 image
    result = results[0]  
    boxes = result.boxes.xyxy.cpu().numpy()        # [xmin, ymin, xmax, ymax]
    scores = result.boxes.conf.cpu().numpy()      # confidence
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    # Fill images JSON
    image_data = {
        "image_id": img_file,
        "filename": "xview_test/"+img_path.split('/')[-1],
        "num_objects": len(boxes),
        "width": int(result.orig_shape[1]),
        "height": int(result.orig_shape[0])
    }
    predictions_data["images"][imgs_idx] = image_data
    imgs_idx += 1

    # Fill annotations
    for i in range(len(boxes)):
        bbox = boxes[i]
        cls = class_ids[i]
        conf = float(scores[i])
        annotation_data = {
            "image_id": img_file,
            "category_id": categories[cls],
            "bbox": [int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])],
            "confidence": conf
        }
        predictions_data["annotations"][annos_idx] = annotation_data
        annos_idx += 1

# Save predictions
import json
with open(OUTPUT_JSON, "w") as f:
    json.dump(predictions_data, f)

print(f"✅ Predictions saved to {OUTPUT_JSON}")

✅ Predictions saved to experiment_results/Detection/3_YOLO_tiled_overlap/prediction.json
